# Sample and label frog calls

Pick nights and 10-second segments from your recordings, label them, and export the results to send to the project lead. The rules for labeling are in `PROTOCOL.md`.

**You need:** your recordings (any folder), and `annotator_config/sampling/frogs.yaml` filled in.

**Run it** in JupyterLab, from top to bottom. Rerunning the notebook is safe: it reuses the segments you already have and never adds more unless you ask in step 6.

In [ ]:
import os
import sys
from pathlib import Path

from IPython.display import display
from jupyter_bioacoustic import BioacousticAnnotator

# Work from the repo folder, so every path here and in annotator_config/ starts there.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")
import functions as fx

## 1. Settings

Edit `annotator_config/sampling/frogs.yaml` (where your audio is, how files are named, your time zone), then set your initials below. Your labels are saved to `output/frog_labels_<initials>.csv`.

In [ ]:
ANNOTATOR = "XX"  # your initials

cfg = fx.load_config("annotator_config/sampling/frogs.yaml")

## 2. List recordings

Finds every audio file, reads the site and time from its name, and groups files into nights. Check the table: each site should show the number of nights and hours you expect. Then fill in coordinates and habitat in `local_data/sites.csv`.

In [ ]:
inventory = fx.build_inventory(cfg)

## 3. Choose nights and sample segments

With `pick_nights: auto` the notebook picks the nights for you. With `pick_nights: manual` a picker appears: tick the nights for each site, click **Use these nights**, then run the cell after it.

The next cell then picks five 10-second segments per night and cuts them into `local_data/clips/`. If you've already sampled, both cells just reuse what you have.

In [ ]:
if cfg["pick_nights"] == "manual":
    picker = fx.pick_nights_widget(inventory, cfg)
    display(picker)

In [ ]:
nights = picker.selection() if cfg["pick_nights"] == "manual" else fx.choose_nights(inventory, cfg)
segments = fx.sample_segments(cfg, inventory, nights)

## 4. Label

For each segment, follow the instructions panel in the tool (full rules in `PROTOCOL.md`). Answers save as you go, so you can stop and come back later.

In [ ]:
ann = BioacousticAnnotator(project="annotator_config/projects/frogs.yaml", annotator=ANNOTATOR)
ann.open()

## 5. Check your labels

Shows how many segments are done per site and lists any answers to fix, such as a **Yes** without a window. Find the segment by its `id` in the tool's table, fix it and submit again.

In [ ]:
issues = fx.check_labels(cfg)
issues

## 6. Sample another round (only if needed)

Finished every segment and want more, for example because none had frogs? Set `ADD_ROUND = True`, run the two cells, then go back to step 4 (rerun its cell). New segments appear at the end of the list. Set `ADD_ROUND` back to `False` afterwards.

New rounds use nights you haven't used yet, never repeat a segment, and stop at `max_segments_per_site`. To search only certain hours or months, fill in `focus:` in the settings file first. Focused rounds are used for training only.

In [ ]:
ADD_ROUND = False

if ADD_ROUND and cfg["pick_nights"] == "manual":
    round_picker = fx.pick_nights_widget(inventory, cfg, new_round=True)
    display(round_picker)

In [ ]:
if ADD_ROUND:
    cfg = fx.load_config("annotator_config/sampling/frogs.yaml")  # picks up any focus: changes
    new_nights = (round_picker.selection() if cfg["pick_nights"] == "manual"
                  else fx.choose_nights(inventory, cfg, new_round=True))
    segments = fx.sample_segments(cfg, inventory, new_nights, new_round=True)

## 7. Export

Packs your clips, segments, labels and settings into `exports/<source>_<date>.zip`. Send that file to the project lead. You can export again later; the new zip replaces the old one.

In [ ]:
zip_path = fx.export_source(cfg)